# Validating regulatory edges with Perturb-seq

scCAFM infers a gene regulatory network from **non-targeting control cells**. In this network, `Gene1` is a transcription factor (TF), `Gene2` is a possible target gene, and the score describes the predicted strength of that relationship.

Perturb-seq provides an independent way to examine these predictions. If a predicted TF-to-target edge is biologically meaningful, perturbing the TF may change the target gene's expression. We compare target expression in TF-perturbed cells with target expression in held-out non-targeting cells using the **Wasserstein distance**. A larger distance means that the two expression distributions are more different.

This comparison is useful evidence, but it is not proof of a direct interaction. A perturbation can have indirect or off-target effects, and the distance does not describe whether expression increased or decreased.

## 1. Set up the tutorial

The model files are read from `assets/`. The prepared K562 tutorial dataset is read from `tutorial_data/perturbseq_edge_validation/`. All paths are relative to the repository.

In [ ]:
from pathlib import Path
import warnings

from IPython.display import display
import numpy as np
import pandas as pd
import scanpy as sc
import torch

warnings.filterwarnings(
    "ignore",
    message="Mismatch dtype between input and weight",
)

from sccafm import GRNInferencer, evaluate_perturbseq_grn


REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "pyproject.toml").is_file():
    REPO_ROOT = REPO_ROOT.parent

MODEL_SOURCE = REPO_ROOT / "assets"
DATA_PATH = (
    REPO_ROOT
    / "tutorial_data"
    / "perturbseq_edge_validation"
    / "K562.h5ad"
)

if not DATA_PATH.is_file():
    raise FileNotFoundError(
        f"Place the prepared K562 dataset at: {DATA_PATH}"
    )

print("Tutorial files are ready.")

## 2. Examine the K562 data

K562 is a human myeloid leukaemia cell line with a large collection of measured gene perturbations. For a concise tutorial, the prepared file retains the 613-gene benchmark panel, including 94 TFs with measured perturbations.

The cells are already divided into three groups:

- **Inference controls** are non-targeting cells used to infer the pooled GRN.
- **Validation controls** are separate non-targeting cells used only for comparison.
- **Validation perturbed cells** carry one of the 94 TF perturbations.

Keeping inference and validation cells separate prevents the perturbation responses from influencing the predicted network.

In [ ]:
adata = sc.read_h5ad(DATA_PATH)

required_obs = {"role", "perturbation", "species"}
missing_obs = required_obs.difference(adata.obs.columns)
if missing_obs:
    raise KeyError(
        "The tutorial dataset is missing observation fields: "
        + ", ".join(sorted(missing_obs))
    )

role_counts = adata.obs["role"].value_counts()
n_perturbed_tfs = adata.obs.loc[
    adata.obs["role"] == "validation_perturbed",
    "perturbation",
].nunique()

dataset_overview = pd.DataFrame(
    [
        {
            "Dataset": "K562",
            "Species": "human",
            "Genes": adata.n_vars,
            "Perturbed TFs": n_perturbed_tfs,
            "Inference controls": role_counts["inference_control"],
            "Validation controls": role_counts["validation_control"],
            "Validation perturbed cells": role_counts[
                "validation_perturbed"
            ],
        }
    ]
)
display(
    dataset_overview.style.format(
        {
            "Genes": "{:,.0f}",
            "Perturbed TFs": "{:,.0f}",
            "Inference controls": "{:,.0f}",
            "Validation controls": "{:,.0f}",
            "Validation perturbed cells": "{:,.0f}",
        }
    )
)

## 3. Prepare the expression data

The tutorial file contains expression values normalized and log-transformed by the established Perturb-seq pipeline. We keep those values for validation.

For scCAFM inference, we copy only the inference controls, reverse the log transformation, and normalize every cell to a total expression of 10000. The original `adata` object and the file on disk are not modified. Gene filtering is unnecessary here because the tutorial file already contains the fixed 613-gene panel.

In [ ]:
inference_adata = adata[
    adata.obs["role"] == "inference_control"
].copy()
validation_adata = adata[
    adata.obs["role"].isin(
        ["validation_control", "validation_perturbed"]
    )
].copy()

inference_adata.X = np.expm1(
    np.asarray(inference_adata.X, dtype=np.float32)
)
sc.pp.normalize_total(inference_adata, target_sum=10000)

prepared_overview = pd.DataFrame(
    [
        {
            "Inference cells": inference_adata.n_obs,
            "Validation cells": validation_adata.n_obs,
            "Genes": inference_adata.n_vars,
            "Expression total for inference": 10000,
        }
    ]
)
display(
    prepared_overview.style.format(
        {
            "Inference cells": "{:,.0f}",
            "Validation cells": "{:,.0f}",
            "Genes": "{:,.0f}",
            "Expression total for inference": "{:,.0f}",
        }
    )
)

## 4. Infer a pooled GRN

The K562 inference cells represent one non-targeting control population, so we average their cell-specific networks to obtain one pooled GRN. The model is loaded once on one GPU.

This tutorial uses `attention_backend="fa2"`. FA2 changes how attention is computed efficiently; it does not change the definition of the inferred GRN. We keep the pooled GRN unfiltered because edge selection is performed after self-edges are removed in the validation step.

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("This tutorial requires one CUDA GPU.")

inferencer = GRNInferencer.from_pretrained(
    MODEL_SOURCE,
    device="cuda:0",
    attention_backend="fa2",
    max_length=1024,
    species_key="species",
)

pooled_grn = inferencer.infer_pooled(
    inference_adata,
    batch_size=8,
    score_threshold=None,
    top_k_edges=None,
)

pooled_overview = pd.DataFrame(
    [
        {
            "Cells pooled": pooled_grn.n_cells,
            "Source TFs": pooled_grn.shape[0],
            "Target genes": pooled_grn.shape[1],
            "Non-self candidate edges": (
                pooled_grn.shape[0] * (pooled_grn.shape[1] - 1)
            ),
        }
    ]
)
display(
    pooled_overview.style.format(
        {
            "Cells pooled": "{:,.0f}",
            "Source TFs": "{:,.0f}",
            "Target genes": "{:,.0f}",
            "Non-self candidate edges": "{:,.0f}",
        }
    )
)

## 5. Validate the top regulatory edges

We now remove self-edges and select the 100 highest-scoring TF-to-target predictions. The Perturb-seq cells are not used to choose these edges.

For each selected edge, `evaluate_perturbseq_grn()` compares the target gene between held-out non-targeting cells and cells where the source TF was perturbed. The result table keeps the scCAFM rank and score alongside the observed Wasserstein distance.

In [ ]:
TOP_K_EDGES = 100

evaluation = evaluate_perturbseq_grn(
    pooled_grn,
    validation_adata,
    perturbation_key="perturbation",
    control_label="non-targeting",
    top_k_edges=TOP_K_EDGES,
)
validated_edges = evaluation.to_edge_table()

validation_summary = pd.DataFrame(
    [
        {
            "Candidate edges": evaluation.n_candidates,
            "Evaluated edges": evaluation.n_evaluated_edges,
            "Perturbed TFs": evaluation.n_perturbed_tfs,
            "Validation controls": evaluation.n_control_cells,
            "Validation perturbed cells": evaluation.n_perturbed_cells,
            "Mean distance": evaluation.mean_wasserstein_distance,
            "Median distance": evaluation.median_wasserstein_distance,
        }
    ]
)
display(
    validation_summary.style.format(
        {
            "Candidate edges": "{:,.0f}",
            "Evaluated edges": "{:,.0f}",
            "Perturbed TFs": "{:,.0f}",
            "Validation controls": "{:,.0f}",
            "Validation perturbed cells": "{:,.0f}",
            "Mean distance": "{:.4f}",
            "Median distance": "{:.4f}",
        }
    )
)

display(
    validated_edges.head(10).style.format(
        {
            "rank": "{:,.0f}",
            "score": "{:.4f}",
            "n_control": "{:,.0f}",
            "n_perturbed": "{:,.0f}",
            "wasserstein_distance": "{:.4f}",
        }
    )
)

## 6. Optional: save the validation table

The cell below is disabled by default. When enabled, it saves all 100 evaluated edges with these columns:

```text
rank,Gene1,Gene2,score,n_control,n_perturbed,wasserstein_distance
```

You may change `TOP_K_EDGES` before evaluation to examine a larger or smaller ranked set. A larger set includes weaker model predictions and takes longer to validate.

In [ ]:
SAVE_VALIDATED_EDGES = True

if SAVE_VALIDATED_EDGES:
    output_dir = REPO_ROOT / "results" / "perturbseq_edge_validation"
    output_dir.mkdir(parents=True, exist_ok=True)
    output_path = output_dir / f"K562_top{TOP_K_EDGES}_validated_edges.csv"
    if output_path.exists():
        raise FileExistsError(
            f"Refusing to overwrite the existing file: {output_path}"
        )
    validated_edges.to_csv(output_path, index=False)
    print(f"Saved validated edges to {output_path}")
else:
    print(
        "CSV export is disabled. Set SAVE_VALIDATED_EDGES = True "
        "to save the table."
    )

## What you learned

`pooled_grn` is the regulatory network inferred only from non-targeting K562 cells. `validated_edges` contains its top 100 non-self predictions together with an independent Perturb-seq response for each TF-to-target pair.

The scCAFM score and Wasserstein distance answer different questions: the score ranks the model's predicted edges, whereas the distance summarizes the observed expression shift after perturbing the source TF. They should be interpreted together rather than treated as the same quantity.